# 실습 6. 모듈러 RAG 파이프라인 — `pipeline(question)` 한 함수로 조립

## 실습 목표

Advanced RAG 구성요소를 **'모듈'로 조립**해 하나의 `pipeline(question)` 함수로 만듭니다(교안 6장).
BDAI `rag07_full_pipeline` 의 **흐름**을 그대로 따릅니다.

```
질의
  │
  ├─ ① Multi-Query (LLM 변형)            ← 실습4
  ▼
② Hybrid Search (BM25 + Vector, RRF)     ← 실습3
  │
  ▼
③ CrossEncoder Rerank (Top-3)            ← 실습5
  │
  ▼
④ LLM 최종 답변 (문맥에만 근거)
```

- **recall(넓게) → precision(좁게)** 의 2단 깔때기
- 각 모듈은 인터페이스만 맞으면 **교체 가능** → 모듈러 RAG의 가치는 **실험 속도**

> LLM 호출 + Cross-Encoder 사용 → `MLAPI_*` 와 리랭커 모델 필요.

## 1. 준비 — 모든 모듈 헬퍼 로드

In [2]:
from _common import (rag_tech_documents, build_vector_retriever, build_bm25,
                     rrf_fuse, multi_query, rerank, get_llm)

docs = rag_tech_documents()
vec = build_vector_retriever(docs, k=10, name="d21_modular_nb")
bm25 = build_bm25(docs, k=10)
llm = get_llm()

def topics(ds): return [d.metadata.get("topic", "?") for d in ds]

print("모듈 헬퍼 로드 완료 · 코퍼스", len(docs), "문서")

/Users/bkk/프로젝트/yeardream/.venv-1/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7900.00it/s]


모듈 헬퍼 로드 완료 · 코퍼스 10 문서


## 2. `pipeline(question)` — 흐름을 한 함수로 (교안 6.2)

`rag07` 흐름을 그대로 옮깁니다: ① 멀티쿼리로 질의 확장 → ② 각 변형을 벡터+BM25 검색 후 RRF
융합(후보 6) → ③ Cross-Encoder 로 top-3 재정렬 → ④ 그 문맥에만 근거해 LLM 답변.
각 단계 산출물을 `dict` 로 반환해 **어느 모듈이 무엇을 했는지** 보이게 합니다.

In [3]:
from langchain_core.messages import SystemMessage, HumanMessage

def pipeline(question: str, verbose: bool = True) -> dict:
    # ① Multi-Query: 질의를 여러 표현으로 확장(원 질문 포함)
    variants = multi_query(llm, question, n=3)
    # ② Hybrid Search: 각 변형 벡터검색 + 원 질문 BM25 → RRF 융합(후보 6)
    lists = [vec.invoke(v) for v in variants] + [bm25.invoke(question)]
    candidates = rrf_fuse(lists, top_n=6)
    # ③ Rerank: Cross-Encoder 로 정밀 재정렬(top-3)
    reranked = rerank(question, candidates, top_n=3)
    # ④ Generate: 검색 문맥에만 근거해 답변
    context = "\n".join(f"({d.metadata['topic']}) {d.page_content}" for d in reranked)
    answer = llm.invoke([
        SystemMessage(content="제공된 [문맥]에만 근거해 한국어로 간결히 답해. 문맥에 없으면 '찾을 수 없습니다'."),
        HumanMessage(content=f"[문맥]\n{context}\n\n[질문] {question}"),
    ]).content
    if verbose:
        print(f"① Multi-Query 변형 {len(variants)}개")
        print(f"② Hybrid 후보(6) : {topics(candidates)}")
        print(f"③ Rerank top-3   : {topics(reranked)}")
        print(f"④ 답변           : {answer[:120]}…")
    return {"variants": variants, "candidates": candidates,
            "reranked": reranked, "answer": answer}

print("pipeline() 정의 완료")

pipeline() 정의 완료


## 3. 파이프라인 실행 — baseline(벡터 단독)과 비교

벡터 단독 top-3 와, 전체 파이프라인의 최종 top-3·답변을 비교합니다.

In [4]:
QUERY = "검색 품질을 종합적으로 끌어올리는 방법은?"
print("baseline 벡터 단독 top3:", topics(vec.invoke(QUERY)[:3]))
print("─" * 60)
out = pipeline(QUERY)

baseline 벡터 단독 top3: ['청킹', '하이브리드', '쿼리변환']
────────────────────────────────────────────────────────────


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 13533.34it/s]


① Multi-Query 변형 4개
② Hybrid 후보(6) : ['청킹', '하이브리드', 'BM25', '쿼리변환', '리랭킹', '벡터검색']
③ Rerank top-3   : ['벡터검색', '리랭킹', '쿼리변환']
④ 답변           : 문맥에 따르면, 검색 품질을 종합적으로 끌어올리려면 다음 조합이 효과적입니다.

1. **벡터 검색**으로 1차 후보를 찾고  
2. **MultiQueryRetriever**로 질의를 여러 표현으로 바꿔 검색 누락…


**포인트 — 흐름이 곧 설계**

- 회수(recall)는 **①멀티쿼리 + ②하이브리드 융합**으로 넓히고, 정밀도(precision)는 **③리랭킹**으로 좁힌다
- 마지막 **④생성**은 좁혀진 문맥에만 근거 → 환각↓
- `pipeline()` 한 함수가 곧 **흐름의 명세** — 모듈을 끼고 빼며 효과를 측정(모듈러 RAG = 실험 속도)
- (LLM 비결정성으로 변형·후보·최종은 실행마다 조금 달라질 수 있음)

# [TODO]

## [TODO] 1. 모듈 on/off — 리랭킹 빼고 비교

`pipeline` 의 ③ 리랭킹을 빼고 **②하이브리드 후보의 상위 3개를 그대로** 문맥으로 답변을
생성하는 `pipeline_no_rerank(question)` 를 만들어, 최종 top-3 주제를 출력하세요.

In [8]:
def pipeline_no_rerank(question):
    variants = multi_query(llm, question, n=3)
    lists = [vec.invoke(v) for v in variants] + [bm25.invoke(question)]
    candidates = rrf_fuse(lists, top_n=6)
    final = candidates[:3]
    context = "\n".join(f"({d.metadata['topic']}) {d.page_content}" for d in final)
    answer = llm.invoke([
        SystemMessage(content="제공된 [문맥]에만 근거해 한국어로 간결히 답해. 문맥에 없으면 '찾을 수 없습니다'."),
        HumanMessage(content=f"[문맥]\n{context}\n\n[질문] {question}"),
    ]).content
    print(f"리랭킹 OFF 답변: {answer[:120]}…")
    return topics(final)

print("리랭킹 OFF 최종 top3:", pipeline_no_rerank(QUERY))

리랭킹 OFF 답변: 검색 품질을 종합적으로 높이려면:

1. **청크 분할 전략 최적화**
   - 너무 잘게 자르지 말고, 너무 크게 잡지도 않기
   - 보통 **500~1500 토큰** 범위에서 **의미 단위로 분할**
   - …
리랭킹 OFF 최종 top3: ['청킹', '하이브리드', 'BM25']


## [TODO] 2. 다른 질문으로 파이프라인 실행

`pipeline("BM25와 벡터 검색의 차이는 무엇인가?")` 을 실행하고, **최종 답변(`out["answer"]`)** 을
출력하세요. (정답 문서는 doc01 벡터검색·doc02 BM25 — 둘 다 검색돼 비교 답변이 나옵니다)

In [10]:
# [TODO] 2: 새 질문으로 pipeline 실행 후 답변 출력
# [TODO] 여기에 코드를 작성하세요.
out = pipeline("BM25와 벡터 검색의 차이는 무엇인가?")

out["answer"]

① Multi-Query 변형 4개
② Hybrid 후보(6) : ['벡터검색', 'BM25', '하이브리드', '리랭킹', 'LangChain마이그레이션', '쿼리변환']
③ Rerank top-3   : ['BM25', '하이브리드', '벡터검색']
④ 답변           : BM25는 **단어 빈도와 문서 길이**를 바탕으로 점수를 매기는 **키워드 기반 검색**입니다.  
- **장점:** 정확한 단어 일치에 강함, 인덱싱이 빠름  
- **단점:** 동의어나 의미 유사성은 잘 잡지 …


'BM25는 **단어 빈도와 문서 길이**를 바탕으로 점수를 매기는 **키워드 기반 검색**입니다.  \n- **장점:** 정확한 단어 일치에 강함, 인덱싱이 빠름  \n- **단점:** 동의어나 의미 유사성은 잘 잡지 못함\n\n벡터 검색은 텍스트를 **임베딩 벡터**로 바꾼 뒤 **코사인 유사도** 등으로 가까운 문서를 찾는 **의미 기반 검색**입니다.  \n- **장점:** 단어가 달라도 의미가 비슷하면 검색 가능  \n- **단점:** 정확한 키워드 일치에는 약함\n\n즉, **BM25는 정확한 단어 매칭에 강하고**, **벡터 검색은 의미 유사성 검색에 강하다**는 차이가 있습니다.'

## 실습 정리

- Advanced RAG 구성요소를 **`pipeline()` 한 함수**로 조립(멀티쿼리→하이브리드→리랭킹→생성)
- 흐름 = recall(넓게)→precision(좁게)→생성(문맥 근거) — **흐름이 곧 설계**
- 모듈 on/off 로 효과 측정 → 모듈러 RAG의 가치는 **실험 속도**
- 검색 + 생성을 연결해 **end-to-end RAG** 완성